In [ ]:
"""
Step 00: Data Quality Check (run before step01 and whenever raw data is added or corrected)

Objective:

    - Check the raw IBKR minute bars before any feature or target is generated.
    - Candlestick consistency (high/low around the body, positive prices, duplicated timestamps).
    - Bad ticks: wicks more than 3% beyond the bar body and the neighbouring bars (so/core/bad_ticks.py; expected 0
      once the one-off correction of 2026-10-04 has been applied).
    - Bars per session compared with the NYSE schedule (390 on a full day, fewer on half days).
    - Sessions present in the schedule but missing from the data (e.g. 2007-07-02, not available from IBKR).

Every rolling window in steps 01 / 02 / 03 counts ROWS, not minutes, so missing minutes silently change the meaning
of the indicators. This report documents how often that happens.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT PLOTLY EXPRESS
import plotly.express as px
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION
from so import config
# IMPORT TRANSFORM OHLCV DATA FUNCTIONS
from so.features.ohlcv_data_utils import format_ohlcv_pdf, get_date_ohlcv_DQ_info_pdf, get_ohlcv_sanity_issue_pdf
# IMPORT MARKET DATETIME FUNCTIONS
from so.core.datetime_utils import ny_tz, get_date_range_market_schedule_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_path_file_list, read_pdf_from_csv_file_path_list, write_csv_file_to_path
# IMPORT THE BAD TICK RULE (ONE-OFF RAW DATA CORRECTION, 2026-10-04)
from so.core.bad_ticks import get_bad_tick_pdf, BAD_TICK_THRESHOLD
# IMPORT PATHS
from so import paths

In [ ]:
# CALL FUNCTION TO GET ALL FILES IN PATH
rawzone_data_file_path_list = get_path_file_list(paths.LOCAL_OHLCV_DATA_FILE_PATH_STR)[:]
# CALL FUNCTION TO READ PDF FROM PATH LIST
raw_ohlcv_pdf = read_pdf_from_csv_file_path_list(rawzone_data_file_path_list)
# CORRECT THE TIMESTAMP DATE COLUMN
raw_ohlcv_pdf["timestamp"] = pd.to_datetime(raw_ohlcv_pdf.timestamp, utc=True).dt.tz_convert(ny_tz)
raw_ohlcv_pdf["date"] = raw_ohlcv_pdf["timestamp"].dt.date
# DISPLAY INFORMATION
print(f"Raw Row Count:\t{len(raw_ohlcv_pdf):,}")
print(f"min timestamp:\t{raw_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{raw_ohlcv_pdf.timestamp.max()}")
# PREVIEW DATAFRAME
raw_ohlcv_pdf

In [ ]:
# CALL FUNCTION TO COLLECT THE CANDLESTICK SANITY ISSUES (ON THE RAW DATA, BEFORE DUPLICATES ARE REJECTED)
sanity_issue_pdf = get_ohlcv_sanity_issue_pdf(raw_ohlcv_pdf)
# DISPLAY INFORMATION
print(f"Rows With Issues:\t{len(sanity_issue_pdf):,}")
# PREVIEW THE ISSUE COUNTS
display(sanity_issue_pdf["issue"].str.split(",").explode().value_counts().reset_index()) if not sanity_issue_pdf.empty else None
# PREVIEW DATAFRAME
sanity_issue_pdf

In [ ]:
# CALL FUNCTION TO COLLECT THE BAD TICKS (WICKS BEYOND THE BODY AND THE NEIGHBOURS; SEE so/core/bad_ticks.py)
bad_tick_pdf = get_bad_tick_pdf(raw_ohlcv_pdf, BAD_TICK_THRESHOLD)
# DISPLAY INFORMATION (0 EXPECTED AFTER 'python so/core/bad_ticks.py --apply')
print(f"Bad Ticks (> {BAD_TICK_THRESHOLD:.0%}):\t{len(bad_tick_pdf):,}")
# PREVIEW DATAFRAME
bad_tick_pdf

In [ ]:
# CALL FUNCTION TO COLLECT DATA (REJECTS EXACT DUPLICATE ROWS)
complete_ohlcv_pdf = format_ohlcv_pdf(raw_ohlcv_pdf.copy()).sort_values("timestamp").reset_index(drop=True)
# CALL FUNCTION TO COMPARE THE BARS PER SESSION WITH THE MARKET SCHEDULE (-1 LESS DATA, 0 NORMAL, 1 MORE DATA)
date_DQ_info_pdf = get_date_ohlcv_DQ_info_pdf(complete_ohlcv_pdf)
# DISPLAY THE STATUS COUNTS
display(date_DQ_info_pdf["status"].value_counts().rename({-1: "less data", 0: "complete", 1: "more data"}).reset_index())
# PREVIEW THE INCOMPLETE SESSIONS
incomplete_session_pdf = date_DQ_info_pdf[date_DQ_info_pdf["status"] != 0].copy()
incomplete_session_pdf["missing_bar_count"] = incomplete_session_pdf["minutes_open"] - incomplete_session_pdf["ts_count"]
# PREVIEW DATAFRAME
incomplete_session_pdf.sort_values("missing_bar_count", ascending=False)

In [ ]:
# COLLECT THE MARKET SCHEDULE OF THE DATA RANGE
market_schedule_pdf = get_date_range_market_schedule_pdf(str(complete_ohlcv_pdf["date"].min()), str(complete_ohlcv_pdf["date"].max()))
# COLLECT THE EXPECTED AND PRESENT SESSION DATES
expected_date_set = set(pd.to_datetime(market_schedule_pdf["date"]).dt.date)
present_date_set = set(complete_ohlcv_pdf["date"].unique())
# COLLECT THE MISSING AND UNEXPECTED SESSIONS
missing_session_list = sorted(expected_date_set - present_date_set)
unexpected_session_list = sorted(present_date_set - expected_date_set)
# DISPLAY INFORMATION
print(f"Expected Sessions:\t{len(expected_date_set):,}")
print(f"Present Sessions:\t{len(present_date_set):,}")
print(f"Missing Sessions:\t{missing_session_list}")
print(f"Unexpected Sessions:\t{unexpected_session_list}")

In [ ]:
# CREATE A FIGURE OF THE BARS PER SESSION
fig = px.scatter(date_DQ_info_pdf, x="date", y="ts_count", color=date_DQ_info_pdf["status"].astype(str),
                 title="Bars Per Session (status: -1 less data, 0 complete, 1 more data)", template="plotly_dark", width=1100, height=500)
# DISPLAY FIGURE
fig.show()

In [ ]:
# SAVE THE DATA QUALITY REPORTS
write_csv_file_to_path(date_DQ_info_pdf, f"{paths.LOCAL_DQ_REPORT_DATA_FILE_PATH_STR}date_DQ_info_data.csv", "W")
write_csv_file_to_path(sanity_issue_pdf, f"{paths.LOCAL_DQ_REPORT_DATA_FILE_PATH_STR}sanity_issue_data.csv", "W")
write_csv_file_to_path(bad_tick_pdf, f"{paths.LOCAL_DQ_REPORT_DATA_FILE_PATH_STR}bad_tick_data.csv", "W")